# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/boluwatifeakintayo/boluwatife-FLYRANKAI-repo/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import userdata
hf_token = userdata.get('flyrank-w3')
print("Token loaded:", hf_token[:8] + "..." if hf_token else "MISSING")

Token loaded: hf_PItrT...


In [2]:
import duckdb

con = duckdb.connect()

con.sql("INSTALL httpfs")
con.sql("LOAD httpfs")

# Newer DuckDB versions use CREATE SECRET instead of SET hf_token
# This registers your token as a named "secret" DuckDB can use when it sees hf:// paths
con.sql(f"""
    CREATE SECRET hf_secret (
        TYPE huggingface,
        TOKEN '{hf_token}'
    )
""")

# Now list what's actually in the dataset
result = con.sql("""
    SELECT * FROM glob('hf://datasets/FlyRank/internship-warehouse/**')
""").df()
print(result)

                                                 file
0   hf://datasets/FlyRank/internship-warehouse/.gi...
1   hf://datasets/FlyRank/internship-warehouse/REA...
2   hf://datasets/FlyRank/internship-warehouse/dim...
3   hf://datasets/FlyRank/internship-warehouse/dim...
4   hf://datasets/FlyRank/internship-warehouse/fac...
5   hf://datasets/FlyRank/internship-warehouse/fac...
6   hf://datasets/FlyRank/internship-warehouse/fac...
7   hf://datasets/FlyRank/internship-warehouse/fac...
8   hf://datasets/FlyRank/internship-warehouse/fac...
9   hf://datasets/FlyRank/internship-warehouse/fac...
10  hf://datasets/FlyRank/internship-warehouse/fac...
11  hf://datasets/FlyRank/internship-warehouse/fac...
12  hf://datasets/FlyRank/internship-warehouse/fac...
13  hf://datasets/FlyRank/internship-warehouse/fac...
14  hf://datasets/FlyRank/internship-warehouse/fac...
15  hf://datasets/FlyRank/internship-warehouse/fac...
16  hf://datasets/FlyRank/internship-warehouse/fac...
17  hf://datasets/FlyRank/in

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## 1. Unit of analysis + time window

**One row = one page's performance, on one specific day**, drawn from
`fact_content_daily_performance`, for the month of **March 2026**
(`month=2026-03`).

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [3]:
march = con.sql("""
    SELECT *
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
    LIMIT 5
""").df()

print(march.columns.tolist())
march.head(5)
# Print every column name, one per line, so nothing gets truncated
for col in march.columns:
    print(col)

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']
report_date
client_hash_id
content_hash_id
client_has_gsc
client_has_ga4
gsc_data_available
ga4_data_available
gsc_impressions
gsc_clicks
gsc_sum_position
gsc_avg_position
ga4_pageviews
ga4_sessions
ga4_users
ga4_engaged_sessions
ga4_total_engagement_sec
sessions_organic
sessions_direct
sessions_referral
sessions_social
sessions_paid
sessions_ai
ai_chatgpt
ai_perplexity
ai_gemini
ai_copilot
ai_claude
ai_meta
ai_other
scroll_events
month


## Fields: feature / label / context / excluded

**Context:** `report_date`, `client_hash_id`, `content_hash_id`, `client_has_gsc`,
`client_has_ga4`, `gsc_data_available`, `ga4_data_available`, `month`

**Label:** Not present as a column in this table — must be derived myself by
comparing a page's aggregated metrics (e.g. `gsc_clicks`, `gsc_impressions`)
across two time periods to construct an `is_declining` signal.

**Feature:** `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `ga4_pageviews`,
`ga4_sessions`, `ga4_users`, `ga4_engaged_sessions`, `ga4_total_engagement_sec`,
`sessions_organic`, `sessions_direct`, `sessions_referral`, `sessions_social`,
`sessions_ai`, `scroll_events`

**Excluded:**
- `gsc_sum_position` — redundant once `gsc_avg_position` is available; keeping
  both risks double-weighting the same signal
- `sessions_paid` — paid traffic isn't organic search performance, a different
  channel outside this lane's question
- `ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `ai_copilot`, `ai_claude`,
  `ai_meta`, `ai_other` — too sparse individually at daily grain for a single
  month; would need a much longer aggregation window to be usable.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [4]:
grain_check = con.sql("""
    SELECT content_hash_id, report_date, COUNT(*) AS row_count
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
    GROUP BY content_hash_id, report_date
    HAVING COUNT(*) > 1
""").df()

print(f"Number of (page, date) pairs with duplicate rows: {len(grain_check)}")
grain_check.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Number of (page, date) pairs with duplicate rows: 0


,content_hash_id,report_date,row_count


In [5]:
span_check = con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT content_hash_id) AS unique_pages,
        MIN(report_date) AS earliest_date,
        MAX(report_date) AS latest_date
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
""").df()

print(span_check)

   total_rows  unique_pages earliest_date latest_date
0     9841378        331437    2026-03-01  2026-03-31


In [6]:
total_rows = con.sql("""
    SELECT COUNT(*) AS total
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
""").df()

available_rows = con.sql("""
    SELECT COUNT(*) AS available
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
    WHERE gsc_data_available IS TRUE
""").df()

print(f"Total rows: {total_rows['total'][0]}")
print(f"Rows with GSC data actually available: {available_rows['available'][0]}")
print(f"Survival rate: {available_rows['available'][0] / total_rows['total'][0]:.1%}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total rows: 9841378
Rows with GSC data actually available: 3611061
Survival rate: 36.7%


## 3. Verify it with queries

**Grain check:** 0 duplicate (page, date) pairs found — confirms one row =
one page on one day, as claimed in Section 1.

**Row count + span:** 9,841,378 total rows, 331,437 unique pages, spanning
exactly 2026-03-01 to 2026-03-31 — confirms the March 2026 time window. Note:
if every page reported every day, we'd expect 10,274,547 rows (331,437 × 31);
the actual count is ~433,000 short, meaning some pages have gaps in their
daily record.

**Availability check:** Filtering with `WHERE gsc_data_available IS TRUE`
reduces the dataset from 9,841,378 rows to 3,611,061 rows — a 36.7% survival
rate. Nearly two-thirds of rows lack usable Search Console data for this
month.

### Five features

1. **`gsc_avg_position`** — knowable at the decision moment because it's the
   page's actual measured ranking position on that day, already recorded.
2. **`gsc_impressions`** — knowable because it's simply how many times the
   page appeared in search that day, already logged.
3. **`gsc_clicks`** — knowable for the same reason — an already-recorded count,
   not a future value.
4. **`ga4_engaged_sessions`** — knowable because it's measured behavior from
   that same day, not a projection.
5. **`sessions_organic`** — knowable because it's a same-day traffic count,
   already recorded at the time of reporting.

In [8]:
# Aggregate total clicks per page for March (our "current" month)
march_agg = con.sql("""
    SELECT content_hash_id, SUM(gsc_clicks) AS clicks_march
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
""").df()

# Aggregate total clicks per page for April (the "outcome" month we're comparing against)
april_agg = con.sql("""
    SELECT content_hash_id, SUM(gsc_clicks) AS clicks_april
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/data_0.parquet'
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
""").df()

# Merge them together on the shared page ID, so each row has both months side by side
merged = march_agg.merge(april_agg, on="content_hash_id", how="inner")

# Build our label: did clicks drop from March to April?
merged["is_declining"] = (merged["clicks_april"] < merged["clicks_march"]).astype(int)

print(merged["is_declining"].value_counts())
merged.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

is_declining
0    114244
1     44305
Name: count, dtype: int64


,content_hash_id,clicks_march,clicks_april,is_declining
0,content_7a105f548d9c6916,7.0,8.0,0
1,content_a3ea9792f793ec72,0.0,2.0,0
2,content_36c36abc7650d7af,6.0,4.0,1
3,content_a7da352b73b02668,13.0,8.0,1
4,content_1855a661b4d36130,1.0,0.0,1


In [9]:
# Build features using ONLY March data — nothing that peeks into April
march_features = con.sql("""
    SELECT
        content_hash_id,
        SUM(gsc_impressions) AS impressions_march,
        SUM(gsc_clicks) AS clicks_march,
        AVG(gsc_avg_position) AS avg_position_march,
        SUM(ga4_engaged_sessions) AS engaged_sessions_march,
        SUM(sessions_organic) AS sessions_organic_march
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
""").df()

# Attach the label we already built
honest_df = march_features.merge(merged[["content_hash_id", "is_declining"]], on="content_hash_id", how="inner")
honest_df = honest_df.fillna(0)

print(honest_df.shape)
honest_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(158549, 7)


,content_hash_id,impressions_march,clicks_march,avg_position_march,engaged_sessions_march,sessions_organic_march,is_declining
0,content_39d7361b4945d504,77.0,0.0,4.074107,0.0,0.0,0
1,content_c03ecafd4c999f15,10849.0,22.0,8.240351,0.0,0.0,0
2,content_e689bc511192751a,61.0,0.0,6.015432,0.0,0.0,0
3,content_7dbc094b799e05a4,705.0,1.0,5.956862,0.0,0.0,1
4,content_40b10da45f4c1cb5,50.0,0.0,12.977513,0.0,0.0,0


In [10]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

feature_cols = ["impressions_march", "clicks_march", "avg_position_march",
                 "engaged_sessions_march", "sessions_organic_march"]

X = honest_df[feature_cols]
y = honest_df["is_declining"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

honest_score = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print(f"Honest ROC-AUC (March-only features): {honest_score:.3f}")

Honest ROC-AUC (March-only features): 0.730


In [13]:
# Build the SAME feature set, but sneak in clicks_april as an extra "feature"
# This is the trap: clicks_april is literally what the label was derived from
leaky_df = honest_df.merge(merged[["content_hash_id", "clicks_april"]], on="content_hash_id", how="inner")

leaky_feature_cols = feature_cols + ["clicks_april"]

X_leak = leaky_df[leaky_feature_cols]
y_leak = leaky_df["is_declining"]

X_train_leak, X_test_leak, y_train_leak, y_test_leak = train_test_split(
    X_leak, y_leak, test_size=0.2, random_state=42
)

leaky_model = LogisticRegression(max_iter=1000)
leaky_model.fit(X_train_leak, y_train_leak)

leaky_score = roc_auc_score(y_test_leak, leaky_model.predict_proba(X_test_leak)[:, 1])

# Remove the leaked column — we NEVER keep this in a real model
del leaky_df
del X_leak, y_leak, X_train_leak, X_test_leak, y_train_leak, y_test_leak, leaky_model

print("Leaked feature removed. Keeping the honest model as the real result:")
print(f"Honest ROC-AUC (March-only features): {honest_score:.3f}")

Leaked feature removed. Keeping the honest model as the real result:
Honest ROC-AUC (March-only features): 0.730


### The leakage trap

I deliberately added `clicks_april` as a feature — the exact column my label
(`is_declining`) was derived from. The score jumped from a realistic 0.730
(March-only features) to a perfect 1.000.

A perfect score is not a good sign — it means the model didn't learn a real
pattern, it found the answer key. `clicks_april` isn't correlated with the
label, it IS the label's source data. This is textbook leakage: using
information from the outcome window (the "future" relative to the decision
point) as an input.

I removed `clicks_april` immediately. The honest, trustworthy number for this
lane is **ROC-AUC = 0.730**, using only March data — information that would
actually be available at the moment a real decision needs to be made.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [7]:
gap_check = con.sql("""
    SELECT
        client_has_gsc,
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS available_rows
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
    GROUP BY client_has_gsc
""").df()

print(gap_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   client_has_gsc  total_rows  available_rows
0            True     9841378       3611061.0


Data limits

This data can never tell us WHY a page's performance changed — only that it
did. Only 36.7% of rows have usable Search Console data this month (verified:
3,611,061 of 9,841,378 rows). Checking `client_has_gsc` ruled out one
explanation — every row in this slice already comes from clients WITH GSC
connected (100% True, no False group exists), so the gap is not caused by
missing integrations. The real cause is more likely daily sync gaps or
non-reporting days, though this hasn't been directly verified with a further
query. Any model built only on `gsc_data_available IS TRUE` rows should be
understood as covering roughly a third of possible page-days, not the full
picture.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.